In [36]:
#Import necessary packages
import xarray as xr
import numpy as np 
import matplotlib.pyplot as plt 
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
from matplotlib.lines import Line2D

In [45]:
foccus_path = '/lustre/storeB/project/fou/hi/foccus/malene/run-anemoi-ocean/ppi/external_checkpoint_inference/Inference_res'
result_path_file = 'lr_v2/lr3-5e-04-st200-se42/2024-03-30_24h_18d28_e011_s049990.nc'
ds = xr.open_dataset(f'{foccus_path}/{result_path_file}')

norkyst_path = '/lustre/storeB/project/fou/hi/foccus/datasets/symlinks/norkystv3-hindcast/2024'
file_path_norkyst ='norkyst800-20240330.nc'
norkyst = xr.open_dataset(f'{norkyst_path}/{file_path_norkyst}').isel(s_rho = -1)

In [3]:
norkyst_resampled = norkyst.resample(time = '3H').mean()

/lustre/storeB/project/fou/hi/foccus/.venv/lib64/python3.11/site-packages/xarray/groupers.py:487: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  self.index_grouper = pd.Grouper(


In [8]:
norkyst_first_skip = norkyst_resampled.isel(time = slice(1, None))

In [9]:
norkyst_first_skip

<xarray.Dataset> Size: 8GB
Dimensions:           (time: 7, Y: 1148, X: 2747, s_w: 41)
Coordinates:
    s_rho             float64 8B -0.004904
  * X                 (X) float64 22kB 0.0 800.0 1.6e+03 ... 2.196e+06 2.197e+06
  * Y                 (Y) float64 9kB 0.0 800.0 1.6e+03 ... 9.168e+05 9.176e+05
  * s_w               (s_w) float64 328B -1.0 -0.96 -0.9208 ... -0.01 0.0
    lon               (Y, X) float64 25MB 8.7 8.706 8.711 ... 18.29 18.31 18.33
    lat               (Y, X) float64 25MB 54.29 54.3 54.31 ... 75.71 75.72 75.73
  * time              (time) datetime64[ns] 56B 2024-03-30T03:00:00 ... 2024-...
Data variables: (12/18)
    Uwind_eastward    (time, Y, X) float32 88MB nan nan nan ... 0.7933 0.8033
    Vwind_northward   (time, Y, X) float32 88MB nan nan nan ... -6.16 -6.16
    zeta              (time, Y, X) float32 88MB nan nan nan ... -1.01 -1.01
    ubar_eastward     (time, Y, X) float32 88MB nan nan nan ... -0.038 -0.038
    vbar_northward    (time, Y, X) float32 88MB nan nan ... -0.06933 -0.06933
    u_eastward        (time, Y, X) float32 88MB 0.0 0.0 0.0 ... -0.034 -0.034
    ...                ...
    hc                (time) float64 56B 100.0 100.0 100.0 ... 100.0 100.0 100.0
    Cs_r              (time) float64 56B -1.673e-05 -1.673e-05 ... -1.673e-05
    Cs_w              (time, s_w) float64 2kB -1.0 -0.9798 ... -6.958e-05 0.0
    h                 (time, Y, X) float32 88MB 10.0 10.0 10.0 ... 138.4 136.3
    projection_stere  (time) float64 56B -2.147e+09 -2.147e+09 ... -2.147e+09
    depth             (time) float64 56B -2.147e+09 -2.147e+09 ... -2.147e+09
Attributes: (12/33)
    id:                      15f95603-12d1-4e0f-8cbe-33946594447f
    naming_authority:        no.met
    operational_status:      scientific
    iso_topic_category:      oceans
    activity_type:           Numerical Simulation
    keywords_vocabulary:     GCMDSK:GCMD Science Keywords:https://gcmd.earthd...
    ...                      ...
    project:                 Norkyst_v3
    license:                 https://spdx.org/licenses/CC-BY-4.0 (CC-BY-4.0)
    title:                   Norkyst-800m - ROMS, Norkyst-800m ocean hindcast...
    summary:                 Norkyst-800m (Norwegian Coast 800m horizontal re...
    title_no:                Hindcast prognoser fra havmodellen Norkyst-800m,...
    summary_no:              NorKyst-800m (Norske kystområder med 800m horiso...

In [7]:
ds_first_skip = ds.isel(time = slice(1,-1))
ds_first_skip

<xarray.Dataset> Size: 732MB
Dimensions:            (X: 1148, Y: 2747, time: 7)
Coordinates:
  * X                  (X) float64 9kB 0.0 800.0 1.6e+03 ... 9.168e+05 9.176e+05
  * Y                  (Y) float64 22kB 0.0 800.0 ... 2.196e+06 2.197e+06
  * time               (time) datetime64[ns] 56B 2024-03-30T03:00:00 ... 2024...
Data variables:
    latitude           (X, Y) float32 13MB ...
    longitude          (X, Y) float32 13MB ...
    salinity_0         (time, X, Y) float32 88MB ...
    temperature_0      (time, X, Y) float32 88MB ...
    u_eastward_0       (time, X, Y) float32 88MB ...
    v_northward_0      (time, X, Y) float32 88MB ...
    h                  (time, X, Y) float32 88MB ...
    sea_mask           (time, X, Y) float32 88MB ...
    f                  (time, X, Y) float32 88MB ...
    river_binary_mask  (time, X, Y) float32 88MB ...

In [25]:
#choose out a variable 
temp_havbris = ds_first_skip.temperature_0.values
temp_norkyst = norkyst_first_skip.temperature.values

In [26]:
#temp_norkyst

In [27]:
print(f'The length of temp norkyst is {len(temp_norkyst)} and for Havbris {len(temp_havbris)}')

The length of temp norkyst is 7 and for Havbris 7


In [28]:
#test if it works to filter out the nans or if I have to fill them in
mask = ~np.isnan(temp_norkyst) & ~np.isnan(temp_havbris)

In [ ]:
rmse_temp = root_mean_squared_error(temp_norkyst[mask], temp_havbris[mask])

In [30]:
rmse_temp

array([0.108078], dtype=float32)

In [ ]:
def rmse2D(truth_path,predicted_path):
    truth = xr.open_dataset({truth_path}, engine = 'h5netcdf').isel(s_rho = -1)
    predicted = xr.open_dataset({predicted_path}, engine = 'h5netcdf')

    #Resample truth datastep 3hrs
    truth_resampled = truth.resample(time = '3H').mean()

    #Remove first time step of datasets and align shapes (24hr range)
    truth_skip_timestep = truth_resampled.isel(time = slice(1,None))
    predicted_skip_timestep = predicted.isel(time=slice(1,-1))

    #select out variable values - thinking temperature, salinity, u and v
    target_vars_norkyst = ['u_eastward', 'v_northward', 'temperature', 'salinity']
    target_vars_havbris = ['u_eastward_0', 'v_northward_0', 'temperature_0', 'salinity_0']

    variable_data_nk = []
    for norkyst_var in target_vars_norkyst:
        variable_data_nk.append(truth_skip_timestep[norkyst_var].values)

    variable_data_hb = []
    for havbris_var in target_vars_havbris:
        variable_data_hb.append(predicted_skip_timestep[havbris_var].values)

    rmse_by_timestep = []
    rmse_total = []
    for nk, hb in zip(variable_data_nk, variable_data_hb):
        #calculating MSE and RMSE using groupby for each timestep
        error = (nk - hb)
        mse_by_timestep = error.groupby('time').mean()
        rmse_by_timestep.append(np.sqrt(mse_by_timestep))

        #Also calculate total RMSE for period - will use sklearn for efficient calculation
        #first filter out nans 
        mask = ~np.isnan(nk) & ~np.isnan(hb)
        rmse_total.append(root_mean_squared_error(nk[mask], hb[mask]))

        #make regression lines for each variable predictions
        y_predict = []
        model = LinearRegression()
        model.fit(hb)
        y_predict.append(model.predict(hb))

    for time_rmse, total_rmse in zip(rmse_by_timestep, rmse_total):
        fig, ax = plt.subplots(2,2, figsize = (12,14))
        ax[0,0].set_title('U-eastward')
        ax[0,0].plot(truth_skip_timestep.time, y_predict)
        ax[0,0].scatter(truth_skip_timestep.time, time_rmse)
        custom_line_u = Line2D([0], [0], color = None, label = f'Total RMSE: {total_rmse}')

        ax[0,1].set_title('V-northward')
        ax[0,1].plot(truth_skip_timestep.time, y_predict)
        ax[0,1].scatter(truth_skip_timestep.time, time_rmse)
        custom_line_v = Line2D([0], [0], color = None, label = f'Total RMSE: {total_rmse}')

        ax[1,0].set_title('Temperature')
        ax[1,0].plot(truth_skip_timestep.time, y_predict)
        ax[1,0].scatter(truth_skip_timestep.time, time_rmse)
        custom_line_temp = Line2D([0], [0], color = None, label = f'Total RMSE: {total_rmse}')

        ax[1,1].set_title('Salinity')
        ax[1,1].plot(truth_skip_timestep.time, y_predict)
        ax[1,1].scatter(truth_skip_timestep.time, time_rmse)
        custom_line_salt = Line2D([0], [0], color = None, label = f'Total RMSE: {total_rmse}')

        plt.legend(handles = [custom_line_u, custom_line_v, custom_line_temp, custom_line_salt])


In [44]:
rmse2D(truth_path=f'{norkyst_path}/{file_path_norkyst}', predicted_path=f'{foccus_path}/{result_path_file}')

TypeError: unhashable type: 'set'